# Phase 3 — Models, Evaluation, Explainability, Compression & Deployment Export
### Deep Learning-Based Predictive Maintenance and RUL Estimation  ·  **TensorFlow / Keras only**

**What this notebook does**
1. Trains **classical baselines** (Ridge, RandomForest, ExtraTrees, Gradient Boosting) on hand-crafted window statistics.
2. Trains **6 Keras architectures** (MLP, LSTM, GRU, BiLSTM, CNN-LSTM, **CNN-BiLSTM-Attention**) with several random seeds each.
3. Chooses the final model **on the validation engines only** (the official test set is never used for choosing).
4. Reports honest **test metrics** (MAE, RMSE, R², NASA score, anomaly precision/recall/F1) and an **over-fitting check**.
5. Explains predictions (**Integrated Gradients** + model-agnostic occlusion), and compresses the model (**distillation + quantisation**).
6. Exports everything the **FastAPI backend and Streamlit dashboard** need.

**How over-fitting is controlled:** held-out validation *engines*, early stopping with best-weight restore, dropout, L2 weight decay,
input-noise augmentation, learning-rate decay, seed ensembles, and a train-vs-validation-vs-test gap check at the end.

**Run order:** Phase 1 → Phase 2 → Phase 3. Needs `pip install tensorflow scikit-learn joblib matplotlib pandas`.

In [ ]:
%%writefile src/rul_core.py
"""
rul_core.py  -  data loading, preprocessing, windowing, metrics, anomaly detection.

Pure NumPy / pandas / scikit-learn (NO TensorFlow here), so every step of Phase 1
and Phase 2 can be run, tested and explained without a deep-learning framework.

Dataset : NASA C-MAPSS turbofan engine degradation (FD001 ... FD004).
Task    : given the last `window` cycles of sensor readings, predict Remaining Useful Life (RUL).
"""
from __future__ import annotations

import io
import json
import re
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
RUL_CAP = 125                      # piece-wise linear RUL target (standard practice for C-MAPSS)
ID_COLS = ["unit_id", "cycle"]
SETTING_COLS = [f"setting_{i}" for i in range(1, 4)]
SENSOR_COLS = [f"sensor_{i}" for i in range(1, 22)]
ALL_COLS = ID_COLS + SETTING_COLS + SENSOR_COLS

# Per-dataset configuration. FD002/FD004 have 6 operating conditions -> need per-condition normalisation.
DATASET_CFG = {
    "FD001": dict(n_conditions=1, window=30, train_stride=1, use_cycle=False, description="1 operating condition, 1 fault mode (HPC)"),
    "FD002": dict(n_conditions=6, window=40, train_stride=2, use_cycle=True, description="6 operating conditions, 1 fault mode"),
    "FD003": dict(n_conditions=1, window=30, train_stride=1, use_cycle=False, description="1 operating condition, 2 fault modes"),
    "FD004": dict(n_conditions=6, window=40, train_stride=2, use_cycle=True, description="6 operating conditions, 2 fault modes (hardest)"),
}

SENSOR_INFO = {  # official C-MAPSS sensor descriptions
    "sensor_1": ("T2", "Total temperature at fan inlet"), "sensor_2": ("T24", "Total temperature at LPC outlet"),
    "sensor_3": ("T30", "Total temperature at HPC outlet"), "sensor_4": ("T50", "Total temperature at LPT outlet"),
    "sensor_5": ("P2", "Pressure at fan inlet"), "sensor_6": ("P15", "Total pressure in bypass duct"),
    "sensor_7": ("P30", "Total pressure at HPC outlet"), "sensor_8": ("Nf", "Physical fan speed"),
    "sensor_9": ("Nc", "Physical core speed"), "sensor_10": ("epr", "Engine pressure ratio"),
    "sensor_11": ("Ps30", "Static pressure at HPC outlet"), "sensor_12": ("phi", "Ratio of fuel flow to Ps30"),
    "sensor_13": ("NRf", "Corrected fan speed"), "sensor_14": ("NRc", "Corrected core speed"),
    "sensor_15": ("BPR", "Bypass ratio"), "sensor_16": ("farB", "Burner fuel-air ratio"),
    "sensor_17": ("htBleed", "Bleed enthalpy"), "sensor_18": ("Nf_dmd", "Demanded fan speed"),
    "sensor_19": ("PCNfR_dmd", "Demanded corrected fan speed"), "sensor_20": ("W31", "HPT coolant bleed"),
    "sensor_21": ("W32", "LPT coolant bleed"),
}
# Groups used by the dashboard (C-MAPSS has no vibration sensor; fan/core speed is the rotating-machinery analogue)
SENSOR_GROUPS = {
    "Temperature": ["sensor_2", "sensor_3", "sensor_4"],
    "Pressure": ["sensor_7", "sensor_11", "sensor_6"],
    "Speed / Vibration proxy": ["sensor_8", "sensor_9", "sensor_13", "sensor_14"],
    "Flow / Fuel": ["sensor_12", "sensor_15", "sensor_20", "sensor_21"],
}

DATA_URLS = [
    "https://data.nasa.gov/docs/legacy/CMAPSSData.zip",
    "https://phm-datasets.s3.amazonaws.com/NASA/6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip",
]
_FILE_RE = re.compile(r"^(train|test|RUL)_FD00[1-4]\.txt$")


# ----------------------------------------------------------------------------------------------
# 1. Data acquisition and loading
# ----------------------------------------------------------------------------------------------
def find_cmapss_files(root) -> dict:
    """Return {filename: path} for every C-MAPSS text file found below `root`."""
    root = Path(root)
    return {p.name: p for p in root.rglob("*.txt") if _FILE_RE.match(p.name)} if root.exists() else {}


def _extract_recursive(zip_bytes: bytes, dest: Path, depth: int = 0) -> None:
    """Extract a zip; if it contains further zips (the NASA mirror does), extract those too."""
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        for name in zf.namelist():
            data = zf.read(name)
            if name.lower().endswith(".zip") and depth < 3:
                _extract_recursive(data, dest, depth + 1)
            elif name.lower().endswith(".txt"):
                (dest / Path(name).name).write_bytes(data)


def ensure_dataset(data_dir, datasets=("FD001", "FD004"), verbose=True) -> dict:
    """
    Make sure the C-MAPSS text files exist in `data_dir`.
    Order: (1) already present -> done, (2) a CMAPSSData*.zip placed in data_dir, (3) download from NASA mirrors.
    Raises a FileNotFoundError with plain instructions if everything fails.
    """
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    need = [f"{p}_{d}.txt" for d in datasets for p in ("train", "test", "RUL")]

    def complete():
        found = find_cmapss_files(data_dir)
        return all(n in found for n in need), found

    ok, found = complete()
    if ok:
        if verbose:
            print(f"Dataset already available in {data_dir}")
        return found

    for z in sorted(data_dir.rglob("*.zip")):                       # (2) user-supplied zip
        try:
            _extract_recursive(z.read_bytes(), data_dir)
            ok, found = complete()
            if ok:
                if verbose:
                    print(f"Extracted dataset from {z.name}")
                return found
        except zipfile.BadZipFile:
            pass

    try:                                                            # (3) download
        import requests
    except ImportError:
        requests = None
    if requests is not None:
        for url in DATA_URLS:
            try:
                if verbose:
                    print("Trying", url)
                r = requests.get(url, timeout=(15, 180))
                r.raise_for_status()
                _extract_recursive(r.content, data_dir)
                ok, found = complete()
                if ok:
                    if verbose:
                        print("Download OK")
                    return found
            except Exception as exc:                                # network problems are expected sometimes
                if verbose:
                    print(f"  failed: {type(exc).__name__}: {str(exc)[:100]}")
    raise FileNotFoundError(
        "C-MAPSS files not found and automatic download failed.\n"
        "FIX (30 seconds): download 'CMAPSSData.zip' from NASA (search: 'NASA C-MAPSS turbofan dataset'),\n"
        f"and put the ZIP (or the extracted .txt files) inside:  {data_dir.resolve()}\n"
        "Then re-run this cell. In Google Colab: use the Files panel (left) -> drag the ZIP into data/raw."
    )


def read_table(path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=r"\s+", header=None).dropna(axis=1, how="all")
    if df.shape[1] != len(ALL_COLS):
        raise ValueError(f"{Path(path).name}: expected {len(ALL_COLS)} columns, found {df.shape[1]}")
    df.columns = ALL_COLS
    df["unit_id"] = df["unit_id"].astype(int)
    df["cycle"] = df["cycle"].astype(int)
    return df.sort_values(["unit_id", "cycle"]).reset_index(drop=True)


def load_subset(files: dict, code: str):
    """Return (train_df, test_df, test_rul_array) for 'FD001'..'FD004'."""
    train = read_table(files[f"train_{code}.txt"])
    test = read_table(files[f"test_{code}.txt"])
    rul = pd.read_csv(files[f"RUL_{code}.txt"], sep=r"\s+", header=None).dropna(axis=1, how="all").iloc[:, 0].to_numpy(float)
    if len(rul) != test["unit_id"].nunique():
        raise ValueError(f"{code}: {len(rul)} RUL labels but {test['unit_id'].nunique()} test engines")
    return train, test, rul


def add_rul(train: pd.DataFrame, cap: float = RUL_CAP) -> pd.DataFrame:
    """RUL = (last cycle of the engine) - (current cycle); capped at `cap` (early life = 'healthy plateau')."""
    out = train.copy()
    out["RUL_raw"] = out.groupby("unit_id")["cycle"].transform("max") - out["cycle"]
    out["RUL"] = out["RUL_raw"].clip(upper=cap)
    return out


def split_engines(unit_ids, val_fraction=0.2, seed=SEED):
    """Split by ENGINE (never by row) so no engine is in both train and validation -> no leakage."""
    rng = np.random.RandomState(seed)
    ids = np.array(sorted(unit_ids))
    rng.shuffle(ids)
    n_val = max(1, int(round(len(ids) * val_fraction)))
    return np.sort(ids[n_val:]), np.sort(ids[:n_val])


# ----------------------------------------------------------------------------------------------
# 2. Preprocessing  (fit on TRAIN engines only)
# ----------------------------------------------------------------------------------------------
class Preprocessor:
    """
    Steps (all parameters are learnt on training engines only and stored as plain numbers -> JSON):
      1. operating-condition detection (k-means on the 3 settings) for multi-condition sets (FD002/FD004)
      2. per-condition z-score of every sensor   (removes the effect of altitude / speed / throttle)
      3. drop sensors that are constant (carry no information)
      4. causal exponential smoothing per engine (denoises without looking into the future)
      5. optional 'cycle' feature (operating hours so far - always known at prediction time)
    """

    def __init__(self, n_conditions=1, smooth_span=5, use_cycle=True, cap=RUL_CAP, window=30):
        self.n_conditions, self.smooth_span, self.use_cycle = n_conditions, smooth_span, use_cycle
        self.cap, self.window = cap, window

    # -- operating conditions -------------------------------------------------
    def _cond(self, df):
        if self.n_conditions == 1:
            return np.zeros(len(df), dtype=int)
        s = (df[SETTING_COLS].to_numpy(float) - self.set_mu) / self.set_sd
        d = ((s[:, None, :] - self.centers[None, :, :]) ** 2).sum(-1)
        return d.argmin(1)

    def fit(self, train_df: pd.DataFrame):
        if self.n_conditions > 1:
            from sklearn.cluster import KMeans
            S = train_df[SETTING_COLS].to_numpy(float)
            self.set_mu, self.set_sd = S.mean(0), S.std(0) + 1e-8
            km = KMeans(self.n_conditions, n_init=10, random_state=SEED).fit((S - self.set_mu) / self.set_sd)
            self.centers = km.cluster_centers_
        cond = self._cond(train_df)
        X = train_df[SENSOR_COLS].to_numpy(float)
        C = self.n_conditions
        mu, sd = np.zeros((C, X.shape[1])), np.ones((C, X.shape[1]))
        for c in range(C):
            m = cond == c
            if m.sum() > 1:
                mu[c], sd[c] = X[m].mean(0), X[m].std(0)
        keep = sd.max(0) > 1e-6
        self.sensors = [s for s, k in zip(SENSOR_COLS, keep) if k]
        self._keep = keep
        self.mu, self.sd = mu, np.where(sd < 1e-6, 1.0, sd)
        self.cycle_scale = float(train_df["cycle"].max())
        return self

    @property
    def features(self):
        return self.sensors + (["cycle"] if self.use_cycle else [])

    @property
    def n_features(self):
        return len(self.features)

    # -- transform --------------------------------------------------------------
    def transform_engine(self, df: pd.DataFrame) -> np.ndarray:
        """df = rows of ONE engine sorted by cycle -> (n_cycles, n_features) float32."""
        cond = self._cond(df)
        X = df[SENSOR_COLS].to_numpy(float)
        Z = ((X - self.mu[cond]) / self.sd[cond])[:, self._keep]
        if self.smooth_span > 1:
            Z = pd.DataFrame(Z).ewm(span=self.smooth_span, adjust=False).mean().to_numpy()
        if self.use_cycle:
            Z = np.column_stack([Z, df["cycle"].to_numpy(float) / self.cycle_scale])
        return Z.astype(np.float32)

    def sensor_part(self, feat: np.ndarray) -> np.ndarray:
        return feat[:, : len(self.sensors)]

    # -- (de)serialisation -----------------------------------------------------
    def to_dict(self):
        d = dict(n_conditions=self.n_conditions, smooth_span=self.smooth_span, use_cycle=self.use_cycle, cap=self.cap,
                 window=self.window, sensors=self.sensors, keep=self._keep.tolist(), mu=self.mu.tolist(),
                 sd=self.sd.tolist(), cycle_scale=self.cycle_scale)
        if self.n_conditions > 1:
            d.update(set_mu=self.set_mu.tolist(), set_sd=self.set_sd.tolist(), centers=self.centers.tolist())
        return d

    @classmethod
    def from_dict(cls, d):
        p = cls(d["n_conditions"], d["smooth_span"], d["use_cycle"], d["cap"], d["window"])
        p.sensors, p._keep = d["sensors"], np.array(d["keep"], dtype=bool)
        p.mu, p.sd, p.cycle_scale = np.array(d["mu"]), np.array(d["sd"]), d["cycle_scale"]
        if p.n_conditions > 1:
            p.set_mu, p.set_sd, p.centers = np.array(d["set_mu"]), np.array(d["set_sd"]), np.array(d["centers"])
        return p


# ----------------------------------------------------------------------------------------------
# 3. Sliding windows
# ----------------------------------------------------------------------------------------------
def pad_front(feat: np.ndarray, window: int) -> np.ndarray:
    """Repeat the first row so that every cycle (even cycle 1) has a full window of history."""
    return np.vstack([np.repeat(feat[:1], window - 1, axis=0), feat])


def engine_windows(feat: np.ndarray, window: int) -> np.ndarray:
    """All windows of one engine: window i ends at cycle i  -> (n_cycles, window, F)."""
    padded = pad_front(feat, window)
    w = np.lib.stride_tricks.sliding_window_view(padded, window, axis=0)   # (n, F, window)
    return np.ascontiguousarray(w.transpose(0, 2, 1))


def build_windows(prep: Preprocessor, df: pd.DataFrame, window: int, stride: int = 1):
    """Training/validation windows for labelled engines. Returns X, y(capped), y_raw, unit, cycle."""
    X, y, yr, u, c = [], [], [], [], []
    for uid, g in df.groupby("unit_id", sort=True):
        g = g.sort_values("cycle")
        W = engine_windows(prep.transform_engine(g), window)
        idx = np.arange(len(g) - 1, -1, -stride)[::-1]              # always keep the last cycle of the engine
        X.append(W[idx]); y.append(g["RUL"].to_numpy()[idx]); yr.append(g["RUL_raw"].to_numpy()[idx])
        u.append(np.full(len(idx), uid)); c.append(g["cycle"].to_numpy()[idx])
    return (np.concatenate(X).astype(np.float32), np.concatenate(y).astype(np.float32),
            np.concatenate(yr).astype(np.float32), np.concatenate(u), np.concatenate(c))


def build_test_windows(prep: Preprocessor, test_df: pd.DataFrame, rul_true: np.ndarray, window: int):
    """One window per test engine: its LAST `window` observed cycles (this is the official C-MAPSS protocol)."""
    X, units, last = [], [], []
    for uid, g in test_df.groupby("unit_id", sort=True):
        g = g.sort_values("cycle")
        X.append(engine_windows(prep.transform_engine(g), window)[-1]); units.append(uid); last.append(int(g["cycle"].max()))
    return np.stack(X).astype(np.float32), np.asarray(rul_true, dtype=np.float32), np.array(units), np.array(last)


# ----------------------------------------------------------------------------------------------
# 4. Metrics
# ----------------------------------------------------------------------------------------------
def nasa_score(y_true, y_pred):
    """Asymmetric PHM08 score: late predictions (pred > true) are penalised more. Lower is better."""
    d = np.asarray(y_pred, float) - np.asarray(y_true, float)
    return float(np.sum(np.where(d < 0, np.exp(-d / 13.0) - 1.0, np.exp(d / 10.0) - 1.0)))


def regression_metrics(y_true, y_pred) -> dict:
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    err = y_pred - y_true
    ss_res, ss_tot = float(np.sum(err ** 2)), float(np.sum((y_true - y_true.mean()) ** 2))
    return {"MAE": float(np.mean(np.abs(err))), "RMSE": float(np.sqrt(np.mean(err ** 2))),
            "R2": 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan"),
            "NASA_score": nasa_score(y_true, y_pred), "Within_10_cycles_%": float(np.mean(np.abs(err) <= 10) * 100)}


def classification_metrics(y_true, y_pred, y_score=None) -> dict:
    from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    out = {"Precision": float(p), "Recall": float(r), "F1": float(f)}
    if y_score is not None and len(np.unique(y_true)) == 2:
        out["ROC_AUC"] = float(roc_auc_score(y_true, y_score))
    return out


# ----------------------------------------------------------------------------------------------
# 5. Classical (non-deep) baselines: hand-crafted window statistics
# ----------------------------------------------------------------------------------------------
def flat_features(X: np.ndarray) -> np.ndarray:
    """(N, T, F) window -> (N, 9F) statistics: last, mean, std, min, max, slope, mean-of-last-5, 10-step and 20-step change."""
    N, T, F = X.shape
    t = np.arange(T, dtype=np.float32) - (T - 1) / 2
    slope = (X * t[None, :, None]).sum(1) / (t ** 2).sum()
    d10 = X[:, -1] - X[:, max(0, T - 10)]
    d20 = X[:, -1] - X[:, max(0, T - 20)]
    return np.concatenate([X[:, -1], X.mean(1), X.std(1), X.min(1), X.max(1), slope, X[:, -5:].mean(1), d10, d20], axis=1)


def flat_feature_names(names):
    return [f"{n}__{s}" for s in ("last", "mean", "std", "min", "max", "slope", "last5", "d10", "d20") for n in names]


# ----------------------------------------------------------------------------------------------
# 6. Anomaly detection (unsupervised): Mahalanobis distance from the engine's HEALTHY behaviour
# ----------------------------------------------------------------------------------------------
class AnomalyDetector:
    """
    Learns the normal (healthy) sensor distribution from the first part of each training engine's life.
    distance = Mahalanobis distance of the current (smoothed, condition-normalised) sensor vector.
    score in [0,1):  r = distance / alarm-distance ;  score = r^2/(1+r^2)  (0.5 == alarm threshold; the alarm distance is
    first set to the healthy 99.5th percentile and then calibrated on validation engines)
    """

    def __init__(self, healthy_fraction=0.25, quantile=0.995):
        self.healthy_fraction, self.quantile = healthy_fraction, quantile

    def fit(self, healthy_rows: np.ndarray):
        self.mu = healthy_rows.mean(0)
        cov = np.cov(healthy_rows.T) + np.eye(healthy_rows.shape[1]) * 1e-3
        self.prec = np.linalg.inv(cov)
        self.thr = float(np.quantile(self.distance(healthy_rows), self.quantile))
        return self

    def distance(self, rows):
        d = rows - self.mu
        return np.sqrt(np.maximum(np.einsum("ij,jk,ik->i", d, self.prec, d), 0.0))

    def score(self, rows):
        r = self.distance(rows) / max(self.thr, 1e-9)
        return r ** 2 / (1.0 + r ** 2)

    def calibrate(self, rows, labels):
        """Choose the alarm threshold that maximises F1 on VALIDATION engines (label 1 = degraded). Returns (F1, threshold)."""
        d, labels = self.distance(rows), np.asarray(labels).astype(bool)
        best = (-1.0, self.thr)
        for t in np.quantile(d, np.linspace(0.05, 0.98, 150)):
            p = d > t
            tp, fp, fn = int((p & labels).sum()), int((p & ~labels).sum()), int((~p & labels).sum())
            f1 = 2 * tp / max(2 * tp + fp + fn, 1)
            if f1 >= best[0]:
                best = (f1, float(t))
        self.thr = best[1]
        return best

    def to_dict(self):
        return dict(mu=self.mu.tolist(), prec=self.prec.tolist(), thr=self.thr)

    @classmethod
    def from_dict(cls, d):
        o = cls(); o.mu, o.prec, o.thr = np.array(d["mu"]), np.array(d["prec"]), d["thr"]
        return o


def warmup_fix(scores, n=20):
    """The smoothing needs ~n cycles to forget its (noisy) first reading, so early anomaly scores are unreliable:
    cap them at the value reached at cycle n."""
    s = np.array(scores, dtype=float).copy()
    if len(s) > n:
        s[:n] = np.minimum(s[:n], s[n])
    return s


def healthy_rows_from_train(prep: Preprocessor, train_df: pd.DataFrame, fraction=0.25) -> np.ndarray:
    rows = []
    for _, g in train_df.groupby("unit_id"):
        f = prep.sensor_part(prep.transform_engine(g.sort_values("cycle")))
        rows.append(f[: max(5, int(len(f) * fraction))])
    return np.vstack(rows)


# ----------------------------------------------------------------------------------------------
# 7. Health indicators used by the dashboard
# ----------------------------------------------------------------------------------------------
def health_percent(rul, cap=RUL_CAP):
    return float(np.clip(rul / cap, 0, 1) * 100)


def status_from_rul(rul):
    return "HEALTHY" if rul >= 70 else ("WARNING" if rul >= 30 else "CRITICAL")


ACTIONS = {
    "HEALTHY": "Normal operation. Continue routine monitoring.",
    "WARNING": "Schedule maintenance inspection within the next operating window.",
    "CRITICAL": "Immediate maintenance required. Remove from service as soon as possible.",
}


def trend_arrow(series, lookback=15, thr=0.05):
    s = np.asarray(series, float)[-lookback:]
    if len(s) < 3:
        return "→"
    slope = np.polyfit(np.arange(len(s)), s, 1)[0] * lookback
    return "↗" if slope > thr else ("↘" if slope < -thr else "→")


def save_json(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, indent=2, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o)))


def load_json(path):
    return json.loads(Path(path).read_text())

In [ ]:
%%writefile src/rul_models.py
"""
rul_models.py  -  models for RUL prediction.   Framework: TensorFlow / Keras ONLY (no PyTorch).

* classical baselines (scikit-learn):  Ridge, RandomForest, ExtraTrees, HistGradientBoosting
* deep models (Keras):  MLP, LSTM, GRU, BiLSTM, CNN-LSTM, CNN-BiLSTM-Attention
* training with early stopping on held-out ENGINES (anti-overfitting), seed ensembles
* Integrated-Gradients explainability, knowledge distillation, quantisation
Only built-in Keras layers are used, so models can be saved/loaded as plain weight files.
"""
from __future__ import annotations

import os
import time
from pathlib import Path

import numpy as np

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    HAS_TF = True
    TF_VERSION = tf.__version__
except Exception as _exc:          # TensorFlow not installed -> classical models still work
    tf = keras = layers = None
    HAS_TF = False
    TF_VERSION = f"not available ({type(_exc).__name__})"

ARCHS = {
    "mlp": "Flatten + 2 dense layers (deep baseline without temporal structure)",
    "lstm": "LSTM -> dense",
    "gru": "GRU -> dense",
    "bilstm": "Bidirectional LSTM -> dense",
    "cnn_lstm": "1D-CNN (local patterns) -> LSTM (temporal dependencies) -> dense",
    "cnn_lstm_attention": "1D-CNN -> BiLSTM -> temporal attention -> dense  (proposed model)",
}


# ------------------------------------------------------------------------------------------
# classical baselines
# ------------------------------------------------------------------------------------------
def fit_classical(F_train, y_train, seed=0, fast=False):
    """Fit the 4 classical regressors on window-statistics features. Returns {name: fitted_model}."""
    from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
    from sklearn.linear_model import Ridge
    n = 100 if fast else 300
    models = {
        "Ridge": Ridge(alpha=10.0),
        # min_samples_leaf / L2 / early stopping keep the tree models regularised (smaller, less memorisation)
        "RandomForest": RandomForestRegressor(n_estimators=n // 2, min_samples_leaf=15, max_features=0.5, n_jobs=-1, random_state=seed),
        "ExtraTrees": ExtraTreesRegressor(n_estimators=n // 2, min_samples_leaf=15, max_features=0.5, n_jobs=-1, random_state=seed),
        "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=800, learning_rate=0.04, max_leaf_nodes=16, min_samples_leaf=100,
                                                              l2_regularization=10.0, early_stopping=True, validation_fraction=0.15,
                                                              n_iter_no_change=30, random_state=seed),
    }
    for m in models.values():
        m.fit(F_train, y_train)
    return models


# ------------------------------------------------------------------------------------------
# Keras architectures
# ------------------------------------------------------------------------------------------
def set_seed(seed: int):
    np.random.seed(seed)
    if HAS_TF:
        try:
            keras.utils.set_random_seed(seed)
        except Exception:
            tf.random.set_seed(seed)


def _attention_pool(x, window):
    """Temporal attention using only built-in layers: score each time step, softmax, weighted sum."""
    score = layers.Dense(1, activation="tanh", name="att_score")(x)          # (B,T,1)
    weights = layers.Softmax(axis=1, name="att_weights")(score)               # (B,T,1) sums to 1 over time
    weighted = layers.Multiply(name="att_weighted")([x, weights])             # (B,T,H)
    pooled = layers.GlobalAveragePooling1D()(weighted)                        # (B,H)  = context / T
    return layers.Rescaling(float(window), name="att_context")(pooled)       # (B,H)  = context vector


def build_model(arch, window, n_features, units=64, filters=64, dropout=0.2, l2=1e-5, noise=0.03, lr=1e-3):
    """Returns a compiled Keras model that maps (window, n_features) -> RUL / cap  (a number in ~[0,1])."""
    if not HAS_TF:
        raise RuntimeError("TensorFlow is not installed:  pip install tensorflow")
    reg = keras.regularizers.l2(l2)
    inp = layers.Input((window, n_features), name="sensor_window")
    x = layers.GaussianNoise(noise, name="noise_augment")(inp)                # regulariser: active only in training
    if arch == "mlp":
        x = layers.Flatten()(x)
        x = layers.Dense(128, activation="relu", kernel_regularizer=reg)(x)
        x = layers.Dropout(dropout)(x)
        x = layers.Dense(64, activation="relu", kernel_regularizer=reg)(x)
    elif arch in ("lstm", "gru"):
        rnn = layers.LSTM if arch == "lstm" else layers.GRU
        x = rnn(units, kernel_regularizer=reg)(x)
        x = layers.Dropout(dropout)(x)
    elif arch == "bilstm":
        x = layers.Bidirectional(layers.LSTM(units, kernel_regularizer=reg))(x)
        x = layers.Dropout(dropout)(x)
    elif arch == "cnn_lstm":
        x = layers.Conv1D(filters, 3, padding="same", activation="relu", kernel_regularizer=reg)(x)
        x = layers.Conv1D(filters, 3, padding="same", activation="relu", kernel_regularizer=reg)(x)
        x = layers.LSTM(units, kernel_regularizer=reg)(x)
        x = layers.Dropout(dropout)(x)
    elif arch == "cnn_lstm_attention":
        x = layers.Conv1D(filters, 3, padding="same", activation="relu", kernel_regularizer=reg)(x)
        x = layers.Conv1D(filters, 3, padding="same", activation="relu", kernel_regularizer=reg)(x)
        x = layers.Bidirectional(layers.LSTM(units // 2, return_sequences=True, kernel_regularizer=reg))(x)
        x = layers.Dropout(dropout)(x)
        x = _attention_pool(x, window)
    elif arch == "student_gru":                                               # tiny model for distillation / edge
        x = layers.GRU(16)(x)
    else:
        raise ValueError(f"unknown architecture '{arch}'. Choose from {list(ARCHS)}")
    x = layers.Dense(32, activation="relu", kernel_regularizer=reg)(x)
    out = layers.Dense(1, name="rul_scaled")(x)
    model = keras.Model(inp, out, name=arch)
    model.compile(optimizer=keras.optimizers.Adam(lr, clipnorm=1.0), loss=keras.losses.Huber(delta=0.1),
                  metrics=[keras.metrics.MeanAbsoluteError(name="mae")])
    return model


def train_model(model, X_tr, y_tr, X_va, y_va, cap=125.0, epochs=60, batch_size=128, patience=10, verbose=0):
    """Train on RUL/cap. Early stopping on validation ENGINES + LR reduction restores the best weights (anti-overfitting)."""
    cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),
           keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=max(2, patience // 3), min_lr=1e-5)]
    return model.fit(X_tr, y_tr / cap, validation_data=(X_va, y_va / cap), epochs=epochs, batch_size=batch_size,
                     callbacks=cbs, verbose=verbose, shuffle=True)


def predict_rul(model, X, cap=125.0):
    return np.clip(model.predict(X, batch_size=1024, verbose=0).ravel() * cap, 0.0, cap)


def save_weights(model, path):
    path = str(path)
    if not path.endswith(".weights.h5"):
        path += ".weights.h5"
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    model.save_weights(path)
    return path


def load_model(arch, window, n_features, weights_path, **kw):
    m = build_model(arch, window, n_features, **kw)
    m.load_weights(str(weights_path))
    return m


# ------------------------------------------------------------------------------------------
# Explainability: Integrated Gradients  (which sensor / time step drove the prediction?)
# ------------------------------------------------------------------------------------------
def integrated_gradients(model, X, steps=24, batch_size=64, baseline=None):
    """
    IG attribution of the model output to every input value.  baseline = all-zeros (= average healthy state
    because inputs are standardised).  Returns array (N, T, F); sum over axis (1,2) ~ f(x) - f(baseline).
    """
    X = np.asarray(X, dtype=np.float32)
    base = np.zeros_like(X) if baseline is None else np.broadcast_to(baseline, X.shape).astype(np.float32)
    out = np.zeros_like(X)
    alphas = (np.arange(steps, dtype=np.float32) + 0.5) / steps               # mid-point rule
    for s in range(0, len(X), batch_size):
        xb, bb = tf.constant(X[s:s + batch_size]), tf.constant(base[s:s + batch_size])
        total = tf.zeros_like(xb)
        for a in alphas:
            xi = bb + a * (xb - bb)
            with tf.GradientTape() as tape:
                tape.watch(xi)
                y = model(xi, training=False)
            total += tape.gradient(y, xi)
        out[s:s + batch_size] = ((xb - bb) * total / steps).numpy()
    return out


# ------------------------------------------------------------------------------------------
# Compression: knowledge distillation + quantisation (edge / IoT deployment)
# ------------------------------------------------------------------------------------------
def distill_student(teacher_pred_scaled, X_tr, y_tr, X_va, y_va, cap=125.0, alpha=0.7, epochs=40, seed=0, verbose=0):
    """Small GRU student learns from  alpha * teacher + (1-alpha) * ground truth  (soft + hard targets)."""
    set_seed(seed)
    student = build_model("student_gru", X_tr.shape[1], X_tr.shape[2], lr=2e-3, noise=0.0)
    target = alpha * teacher_pred_scaled + (1 - alpha) * (y_tr / cap)
    cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)]
    student.fit(X_tr, target, validation_data=(X_va, y_va / cap), epochs=epochs, batch_size=128, callbacks=cbs, verbose=verbose)
    return student


def float16_quantize(model, arch, window, n_features, **kw):
    """Post-training weight quantisation to float16 (always works): returns (model_with_rounded_weights, size_bytes)."""
    q = build_model(arch, window, n_features, **kw)
    ws = model.get_weights()
    q.set_weights([w.astype(np.float16).astype(np.float32) for w in ws])
    return q, int(sum(w.size for w in ws) * 2)


def tflite_dynamic_quantize(model, out_path):
    """Try TensorFlow-Lite dynamic-range int8 quantisation. Returns (size_bytes, path) or (None, reason)."""
    try:
        conv = tf.lite.TFLiteConverter.from_keras_model(model)
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
        conv._experimental_lower_tensor_list_ops = False
        blob = conv.convert()
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        Path(out_path).write_bytes(blob)
        return len(blob), str(out_path)
    except Exception as exc:
        return None, f"{type(exc).__name__}: {str(exc)[:120]}"


def tflite_predict(path, X, cap=125.0):
    interp = tf.lite.Interpreter(model_path=str(path))
    inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
    preds = []
    for i in range(len(X)):
        interp.resize_tensor_input(inp["index"], [1] + list(X.shape[1:]))
        interp.allocate_tensors()
        interp.set_tensor(inp["index"], X[i:i + 1].astype(np.float32))
        interp.invoke()
        preds.append(float(interp.get_tensor(out["index"]).ravel()[0]))
    return np.clip(np.array(preds) * cap, 0, cap)


def latency_ms(predict_fn, X, repeats=5):
    """Mean time (ms) to predict ONE window (batch of 1), measured over a few windows."""
    predict_fn(X[:1])
    t = time.perf_counter()
    for _ in range(repeats):
        for i in range(min(20, len(X))):
            predict_fn(X[i:i + 1])
    return (time.perf_counter() - t) / (repeats * min(20, len(X))) * 1000.0

In [ ]:
%%writefile src/inference.py
"""
inference.py - load the trained pipeline and produce predictions / dashboard artefacts.
Used by the notebooks (Phase 3 export), the FastAPI backend and the Streamlit app.
Works with Keras members (if TensorFlow is installed) and scikit-learn members.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from src.rul_core import (ACTIONS, SENSOR_INFO, AnomalyDetector, Preprocessor, SENSOR_COLS, SETTING_COLS, engine_windows,
                          flat_features, health_percent, warmup_fix, load_json, status_from_rul, trend_arrow)


class Predictor:
    """artifacts/final/<FDxxx>/  ->  callable RUL predictor with anomaly score and explanations."""

    def __init__(self, art_dir):
        self.dir = Path(art_dir)
        self.meta = load_json(self.dir / "meta.json")
        self.prep = Preprocessor.from_dict(load_json(self.dir / "prep.json"))
        self.anom = AnomalyDetector.from_dict(load_json(self.dir / "anomaly.json"))
        self.window, self.cap = self.meta["window"], self.meta["cap"]
        self._members = None

    # ---- model loading --------------------------------------------------------
    def _load_members(self):
        if self._members is not None:
            return
        members, weights = [], []
        for m in self.meta["final_model"]["members"]:
            weights.append(float(m.get("weight", 1.0)))
            if m["kind"] == "sklearn":
                import joblib
                members.append(("sklearn", joblib.load(self.dir / m["file"])))
            else:
                from src import rul_models as RM
                if not RM.HAS_TF:
                    raise RuntimeError("This saved model needs TensorFlow (pip install tensorflow).")
                members.append(("keras", RM.load_model(m["arch"], self.window, self.prep.n_features, self.dir / m["file"], **m.get("kw", {}))))
        self._members, self._weights = members, weights

    def predict_windows(self, X):
        """X (N, window, F) -> RUL estimates (N,) = average of all ensemble members."""
        self._load_members()
        total, out = 0.0, 0.0
        for (kind, model), w in zip(self._members, self._weights):
            if kind == "sklearn":
                p = np.clip(model.predict(flat_features(X)), 0, self.cap)
            else:
                p = np.clip(model.predict(X, batch_size=1024, verbose=0).ravel() * self.cap, 0, self.cap)
            out = out + w * p
            total += w
        return out / total

    # ---- per-engine API ---------------------------------------------------------
    def timeline(self, df_engine: pd.DataFrame) -> pd.DataFrame:
        """Predicted RUL + anomaly score at EVERY cycle of one engine (only past data used at each cycle)."""
        g = df_engine.sort_values("cycle")
        feat = self.prep.transform_engine(g)
        rul = self.predict_windows(engine_windows(feat, self.window))
        anom = warmup_fix(self.anom.score(self.prep.sensor_part(feat)))
        return pd.DataFrame({"cycle": g["cycle"].to_numpy(), "rul_pred": rul, "anomaly_score": anom,
                             "health_pct": np.clip(rul / self.cap, 0, 1) * 100})

    def assess(self, df_raw: pd.DataFrame) -> dict:
        """Raw sensor history of ONE machine (columns: cycle, setting_1..3, sensor_1..21) -> full health assessment."""
        df = df_raw.copy()
        missing = [c for c in ["cycle"] + SENSOR_COLS if c not in df.columns]
        if missing:
            raise ValueError(f"missing columns: {missing[:5]}")
        for c in SETTING_COLS:
            if c not in df.columns:
                df[c] = 0.0
        tl = self.timeline(df)
        last = tl.iloc[-1]
        rul = float(last["rul_pred"])
        status = status_from_rul(rul)
        return {"cycle": int(last["cycle"]), "n_cycles": int(len(tl)), "rul": rul, "health_pct": health_percent(rul, self.cap),
                "anomaly_score": float(last["anomaly_score"]), "anomaly_flag": bool(last["anomaly_score"] > 0.5),
                "status": status, "action": ACTIONS[status], "timeline": tl[["cycle", "rul_pred", "anomaly_score"]]}

    def occlusion_importance(self, X_last, feature_names=None):
        """
        Model-agnostic explanation: replace one feature of the window by its healthy value (0 after
        standardisation) and see how much the predicted RUL RISES. Large rise = that feature is pushing the RUL down.
        Returns (N, F) array of cycles.
        """
        names = feature_names or self.prep.features
        base = self.predict_windows(X_last)
        out = np.zeros((len(X_last), len(names)))
        for j in range(len(names)):
            Xo = X_last.copy()
            Xo[:, :, j] = 0.0
            out[:, j] = self.predict_windows(Xo) - base
        return out


# ------------------------------------------------------------------------------------------------
# artefacts for the dashboard (computed once after training, then the app only reads files)
# ------------------------------------------------------------------------------------------------
def export_dashboard_data(pred: Predictor, test_df: pd.DataFrame, rul_true: np.ndarray, out_dir) -> dict:
    """Per-engine predictions at EVERY cycle (using only past data), fleet table, explanations, raw test sensors."""
    out_dir = Path(out_dir)
    prep, cap = pred.prep, pred.cap
    units = np.sort(test_df["unit_id"].unique())
    tl_rows, fleet_rows, imp_rows, X_last = [], [], [], []
    for uid, true_last in zip(units, rul_true):
        g = test_df[test_df["unit_id"] == uid].sort_values("cycle")
        feat = prep.transform_engine(g)
        W = engine_windows(feat, pred.window)
        rul = pred.predict_windows(W)
        anom = warmup_fix(pred.anom.score(prep.sensor_part(feat)))
        last_cycle = int(g["cycle"].max())
        true_rul = true_last + (last_cycle - g["cycle"].to_numpy())              # true RUL at every cycle of the test history
        tl_rows.append(pd.DataFrame({"unit_id": uid, "cycle": g["cycle"].to_numpy(), "rul_pred": rul, "rul_true": true_rul,
                                     "rul_true_capped": np.minimum(true_rul, cap), "anomaly_score": anom}))
        X_last.append(W[-1])
        r = float(rul[-1]); st = status_from_rul(r)
        fleet_rows.append({"unit_id": int(uid), "cycles_observed": last_cycle, "rul_pred": r, "rul_true": float(true_last),
                           "rul_true_capped": float(min(true_last, cap)), "abs_error": abs(r - min(true_last, cap)),
                           "health_pct": health_percent(r, cap), "anomaly_score": float(anom[-1]), "status": st, "action": ACTIONS[st]})
    X_last = np.stack(X_last)
    contrib = pred.occlusion_importance(X_last)
    for i, uid in enumerate(units):
        for j, f in enumerate(prep.features):
            z = X_last[i, :, j]
            imp_rows.append({"unit_id": int(uid), "feature": f, "label": SENSOR_INFO.get(f, ("cycles run",))[0],
                             "contribution": float(contrib[i, j]), "z_last": float(z[-1]), "trend": trend_arrow(z)})
    timelines, fleet, imp = pd.concat(tl_rows, ignore_index=True), pd.DataFrame(fleet_rows), pd.DataFrame(imp_rows)
    timelines.round(4).to_csv(out_dir / "timelines.csv.gz", index=False)
    fleet.round(4).to_csv(out_dir / "fleet_summary.csv", index=False)
    imp.round(4).to_csv(out_dir / "engine_importance.csv", index=False)
    test_df[["unit_id", "cycle"] + SETTING_COLS + SENSOR_COLS].to_csv(out_dir / "test_sensors.csv.gz", index=False)
    return {"timelines": timelines, "fleet": fleet, "importance": imp}

In [ ]:
%%writefile src/viz.py
"""viz.py - all figures (matplotlib, one consistent dark industrial theme). Used by the notebooks AND the Streamlit app."""
from __future__ import annotations

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyBboxPatch, Wedge

BG, PANEL, GRID, TEXT, MUTED = "#0e1621", "#152231", "#26374a", "#e6edf5", "#8fa3b8"
OK, WARN, CRIT, BLUE, VIOLET = "#2dd4a8", "#f5b83d", "#ef5b5b", "#4ea8ff", "#a78bfa"
STATUS_COLOR = {"HEALTHY": OK, "WARNING": WARN, "CRITICAL": CRIT}
PALETTE = [BLUE, OK, WARN, VIOLET, CRIT, "#7dd3fc", "#fb7185"]


def apply_style():
    plt.rcParams.update({
        "figure.facecolor": BG, "axes.facecolor": PANEL, "savefig.facecolor": BG, "axes.edgecolor": GRID,
        "axes.labelcolor": MUTED, "axes.titlecolor": TEXT, "axes.titleweight": "bold", "axes.titlesize": 12,
        "xtick.color": MUTED, "ytick.color": MUTED, "text.color": TEXT, "grid.color": GRID, "grid.alpha": 0.6,
        "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False, "legend.facecolor": PANEL,
        "legend.edgecolor": GRID, "font.size": 10, "figure.dpi": 100, "savefig.dpi": 150,
    })


apply_style()


def _finish(fig, path=None):
    if path:
        from pathlib import Path
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, bbox_inches="tight")
    return fig


def fig_gauge(value, title, color=None, vmax=100, unit="%", path=None):
    fig, ax = plt.subplots(figsize=(3.2, 2.1))
    ax.axis("off"); ax.set_xlim(-1.2, 1.2); ax.set_ylim(-0.2, 1.2); ax.set_aspect("equal")
    frac = float(np.clip(value / vmax, 0, 1))
    color = color or (OK if frac > 0.55 else WARN if frac > 0.25 else CRIT)
    ax.add_patch(Wedge((0, 0), 1, 0, 180, width=0.22, color=GRID))
    ax.add_patch(Wedge((0, 0), 1, 180 - 180 * frac, 180, width=0.22, color=color))
    txt = f"{value:.0f}{unit}" if vmax > 1.5 else f"{value:.2f}"
    ax.text(0, 0.30, txt, ha="center", va="center", fontsize=21, fontweight="bold", color=TEXT)
    ax.text(0, -0.05, title, ha="center", va="center", fontsize=9, color=MUTED)
    return _finish(fig, path)


def fig_status_donut(counts: dict, path=None):
    fig, ax = plt.subplots(figsize=(4, 3.4))
    order = ["HEALTHY", "WARNING", "CRITICAL"]
    vals = [counts.get(k, 0) for k in order]
    ax.pie(vals, colors=[STATUS_COLOR[k] for k in order], startangle=90, counterclock=False, wedgeprops=dict(width=0.38, edgecolor=BG),
           labels=[f"{k}\n{v}" for k, v in zip(order, vals)], textprops=dict(color=TEXT, fontsize=9))
    ax.set_title("Fleet status"); ax.grid(False)
    return _finish(fig, path)


def fig_rul_timeline(cycles, pred, true_capped=None, cap=125, cutoff=None, title="Predicted RUL over the machine's life", path=None):
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.axhspan(0, 30, color=CRIT, alpha=0.10); ax.axhspan(30, 70, color=WARN, alpha=0.08); ax.axhspan(70, cap + 5, color=OK, alpha=0.06)
    if true_capped is not None:
        ax.plot(cycles, true_capped, color=MUTED, lw=2, ls="--", label="True RUL (capped)")
    ax.plot(cycles, pred, color=BLUE, lw=2.2, label="Predicted RUL")
    if cutoff is not None:
        ax.axvline(cutoff, color=TEXT, lw=1, ls=":"); ax.text(cutoff, cap, " now", color=TEXT, va="top", fontsize=9)
    ax.set_ylim(0, cap + 5); ax.set_xlabel("Operating cycle"); ax.set_ylabel("RUL (cycles)"); ax.set_title(title); ax.legend(loc="upper right")
    return _finish(fig, path)


def fig_anomaly_timeline(cycles, scores, cutoff=None, title="Anomaly score over time", path=None):
    fig, ax = plt.subplots(figsize=(9, 2.8))
    ax.fill_between(cycles, scores, color=VIOLET, alpha=0.25); ax.plot(cycles, scores, color=VIOLET, lw=2)
    ax.axhline(0.5, color=CRIT, ls="--", lw=1.2); ax.text(np.min(cycles), 0.53, "alarm threshold", color=CRIT, fontsize=8)
    if cutoff is not None:
        ax.axvline(cutoff, color=TEXT, lw=1, ls=":")
    ax.set_ylim(0, 1.02); ax.set_xlabel("Operating cycle"); ax.set_ylabel("Anomaly score"); ax.set_title(title)
    return _finish(fig, path)


def fig_sensor_trends(cycles, series: dict, cutoff=None, title="Key sensor trends (standardised per operating condition)", path=None):
    fig, ax = plt.subplots(figsize=(9, 3.4))
    for i, (name, y) in enumerate(series.items()):
        ax.plot(cycles, y, lw=1.8, color=PALETTE[i % len(PALETTE)], label=name)
    if cutoff is not None:
        ax.axvline(cutoff, color=TEXT, lw=1, ls=":")
    ax.set_xlabel("Operating cycle"); ax.set_ylabel("z-score"); ax.set_title(title); ax.legend(ncol=max(1, len(series)), loc="upper left", fontsize=8)
    return _finish(fig, path)


def fig_importance(names, values, title="Why this prediction? (sensor contribution)", top=8, path=None):
    names, values = np.asarray(list(names)), np.asarray(list(values), float)
    idx = np.argsort(-np.abs(values))[:top][::-1]
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.barh(names[idx], values[idx], color=[CRIT if v > 0 else BLUE for v in values[idx]])
    ax.axvline(0, color=MUTED, lw=1); ax.set_xlabel("contribution to a SHORTER remaining life (red) / longer (blue)"); ax.set_title(title, fontsize=10)
    return _finish(fig, path)


def fig_pred_vs_true(y_true, y_pred, title="Predicted vs actual RUL", path=None):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    fig, axs = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [1.3, 1]})
    err = y_pred - y_true
    axs[0].scatter(y_true, y_pred, c=np.where(np.abs(err) <= 10, OK, np.where(np.abs(err) <= 20, WARN, CRIT)), s=26, alpha=0.85, edgecolor="none")
    m = max(y_true.max(), y_pred.max()) + 5
    axs[0].plot([0, m], [0, m], color=MUTED, ls="--", lw=1.2); axs[0].fill_between([0, m], [-10, m - 10], [10, m + 10], color=OK, alpha=0.08)
    axs[0].set_xlabel("True RUL"); axs[0].set_ylabel("Predicted RUL"); axs[0].set_title(title, fontsize=11)
    axs[1].hist(err, bins=24, color=BLUE, alpha=0.9); axs[1].axvline(0, color=TEXT, lw=1)
    axs[1].set_xlabel("error = predicted − true (cycles)"); axs[1].set_title("Error distribution (green band = ±10 cycles)", fontsize=10)
    fig.tight_layout()
    return _finish(fig, path)


def fig_model_comparison(df, metric="Test_RMSE", title=None, lower_better=True, highlight=None, path=None):
    d = df.sort_values(metric, ascending=not lower_better)
    fig, ax = plt.subplots(figsize=(6.2, max(2.8, 0.38 * len(d) + 1)))
    best = highlight or d.iloc[-1]["Model"]
    ax.barh(d["Model"], d[metric], color=[OK if m == best else BLUE for m in d["Model"]])
    for y, v in enumerate(d[metric]):
        ax.text(v, y, f" {v:.3f}" if abs(v) < 5 else f" {v:.1f}", va="center", fontsize=8, color=TEXT)
    if "R2" in metric:
        ax.set_xlim(0, 1.1); ax.axvline(0.75, color=WARN, ls="--", lw=1); ax.text(0.75, len(d) - 0.4, " target 0.75", color=WARN, fontsize=8)
    ax.set_title(title or metric, fontsize=11)
    fig.tight_layout()
    return _finish(fig, path)


def fig_generalization(train_r2, val_r2, test_r2, path=None):
    fig, ax = plt.subplots(figsize=(5, 3))
    labels, vals = ["train", "validation\n(unseen engines)", "official test\n(unseen engines)"], [train_r2, val_r2, test_r2]
    keep = [(l, v) for l, v in zip(labels, vals) if v is not None]
    bars = ax.bar([l for l, _ in keep], [v for _, v in keep], color=[BLUE, VIOLET, OK][: len(keep)], width=0.55)
    for b, (_, v) in zip(bars, keep):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.3f}", ha="center", fontsize=10, fontweight="bold")
    ax.set_ylim(0, 1.1); ax.set_ylabel("R²"); ax.set_title("Generalisation check"); ax.axhline(0.75, color=WARN, ls="--", lw=1)
    return _finish(fig, path)


def fig_training_curves(histories: dict, path=None):
    fig, ax = plt.subplots(figsize=(8, 3.6))
    for i, (name, h) in enumerate(histories.items()):
        ax.plot(h["loss"], color=PALETTE[i % 7], lw=1.3, alpha=0.7); ax.plot(h["val_loss"], color=PALETTE[i % 7], lw=2, ls="--", label=name)
    ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("Huber loss  (solid = train, dashed = validation)")
    ax.set_title("Learning curves"); ax.legend(fontsize=8, ncol=2)
    return _finish(fig, path)


def fig_pipeline(path=None):
    fig, ax = plt.subplots(figsize=(13, 3.4)); ax.axis("off"); ax.set_xlim(0, 13); ax.set_ylim(0, 3.4)
    boxes = [("Industrial\nmachine\n(21 sensors)", BLUE), ("Preprocessing\nper-condition z-score\nsmoothing", BLUE), ("Time windows\n30–40 cycles", BLUE),
             ("Deep model\n1D-CNN → BiLSTM\n→ Attention", VIOLET), ("RUL\nprediction", OK), ("Health +\nAnomaly +\nExplanation", WARN), ("Dashboard\n+ API", CRIT)]
    for i, (t, c) in enumerate(boxes):
        x = 0.15 + i * 1.85
        ax.add_patch(FancyBboxPatch((x, 0.8), 1.55, 1.8, boxstyle="round,pad=0.04,rounding_size=0.12", fc=PANEL, ec=c, lw=2))
        ax.text(x + 0.775, 1.7, t, ha="center", va="center", fontsize=8.6, color=TEXT)
        if i < len(boxes) - 1:
            ax.annotate("", (x + 1.85, 1.7), (x + 1.58, 1.7), arrowprops=dict(arrowstyle="-|>", color=MUTED, lw=1.5))
    ax.text(6.5, 3.05, "Predictive-maintenance pipeline: from raw sensor data to explainable machine-health intelligence", ha="center", fontsize=11, fontweight="bold")
    ax.text(6.5, 0.3, "Baselines (Ridge · RandomForest · ExtraTrees · GradientBoosting) and 6 Keras architectures are compared on the same split",
            ha="center", fontsize=9, color=MUTED)
    return _finish(fig, path)

In [ ]:
%%writefile src/zoo.py
"""
zoo.py - the MODEL ZOO: every selectable model (classical + Keras deep learning + ensembles) with
metrics, per-engine predictions over time and per-engine explanations.  The web UI reads these files, so a user can
pick a model BEFORE predicting and compare deep learning against classical ML on identical engines.

    build_zoo(root, "FD001")                  -> artifacts/zoo/FD001/{index.json, timelines.csv.gz, importance.csv.gz}
    train_deep_suite(root, "FD001")           -> trains Keras models (TensorFlow) into artifacts/models/FD001/
    ZooPredictor(root, "FD001", model_id)     -> live predictions for the API

Framework: TensorFlow / Keras only for deep models (no PyTorch).  Classical models use scikit-learn.
"""
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from src.rul_core import (ACTIONS, RUL_CAP, AnomalyDetector, Preprocessor, engine_windows, flat_features, health_percent,
                          load_json, regression_metrics, save_json, status_from_rul, warmup_fix)

MODEL_KW = dict(units=64, filters=64, dropout=0.2, l2=1e-5, noise=0.03, lr=1e-3)   # must match training
DEEP_ARCHS = ["mlp", "lstm", "gru", "bilstm", "cnn_lstm", "cnn_lstm_attention"]
CLASSICAL = ["Ridge", "RandomForest", "ExtraTrees", "HistGradientBoosting"]
PRETTY = {"mlp": "MLP (dense network)", "lstm": "LSTM", "gru": "GRU", "bilstm": "BiLSTM", "cnn_lstm": "CNN-LSTM",
          "cnn_lstm_attention": "CNN-BiLSTM-Attention", "Ridge": "Ridge regression", "RandomForest": "Random Forest",
          "ExtraTrees": "Extra Trees", "HistGradientBoosting": "Gradient Boosting"}
DESC = {"mlp": "Flatten + dense layers. No notion of time order: the control experiment.",
        "lstm": "Recurrent network that reads the window step by step.",
        "gru": "Lighter recurrent network, similar idea to LSTM.",
        "bilstm": "LSTM reading the window forwards and backwards.",
        "cnn_lstm": "1D convolutions find local sensor patterns, an LSTM models how they evolve.",
        "cnn_lstm_attention": "CNN + BiLSTM + temporal attention: learns which time steps matter. The proposed model.",
        "Ridge": "Linear model on window statistics. The simplest baseline.",
        "RandomForest": "Many decision trees on window statistics.",
        "ExtraTrees": "Randomised trees on window statistics.",
        "HistGradientBoosting": "Boosted trees on window statistics."}


def paths(root, ds):
    root = Path(root)
    return dict(root=root, proc=root / "data" / "processed" / ds, models=root / "artifacts" / "models" / ds,
                final=root / "artifacts" / "final" / ds, zoo=root / "artifacts" / "zoo" / ds)


def has_tensorflow() -> bool:
    try:
        from src import rul_models as RM
        return bool(RM.HAS_TF)
    except Exception:
        return False


# ------------------------------------------------------------------------------------------------
# Training helpers
# ------------------------------------------------------------------------------------------------
def ensure_classical(root, ds, log=print):
    """Make sure the 4 classical models exist on disk (train them if not; deterministic, takes 1-3 minutes)."""
    import joblib
    from src import rul_models as RM
    P = paths(root, ds)
    d = P["models"] / "classical"
    d.mkdir(parents=True, exist_ok=True)
    if all((d / f"{n}.joblib").exists() for n in CLASSICAL):
        return
    z = np.load(P["proc"] / "arrays.npz")
    log(f"[{ds}] training classical models ...")
    for name, m in RM.fit_classical(flat_features(z["X_train"]), z["y_train"], seed=0).items():
        joblib.dump(m, d / f"{name}.joblib", compress=3)


def train_deep_suite(root, ds, archs=None, seeds=3, epochs=60, patience=10, batch=128, force=False, log=print):
    """Train Keras models (TensorFlow) on the Phase-2 arrays and save weights to artifacts/models/<ds>/."""
    from src import rul_models as RM
    if not RM.HAS_TF:
        raise RuntimeError("TensorFlow is not installed.  Run:  pip install tensorflow")
    P = paths(root, ds)
    z = np.load(P["proc"] / "arrays.npz")
    P["models"].mkdir(parents=True, exist_ok=True)
    window, F = z["X_train"].shape[1], z["X_train"].shape[2]
    done = []
    for i, arch in enumerate(archs or DEEP_ARCHS):
        t0 = time.time()
        try:
            for s in range(seeds):
                f = P["models"] / f"{arch}_s{s}.weights.h5"
                if f.exists() and not force:
                    continue
                RM.set_seed(1000 * (DEEP_ARCHS.index(arch) + 1) + s)
                model = RM.build_model(arch, window, F, **MODEL_KW)
                RM.train_model(model, z["X_train"], z["y_train"], z["X_val"], z["y_val"], RUL_CAP, epochs, batch, patience, verbose=0)
                RM.save_weights(model, P["models"] / f"{arch}_s{s}")
            done.append(arch)
            log(f"[{ds}] {PRETTY[arch]:22s} trained ({seeds} seeds, {time.time() - t0:.0f}s)")
        except Exception as exc:                       # one failing architecture must never stop the others
            log(f"[{ds}] {PRETTY[arch]:22s} FAILED: {type(exc).__name__}: {str(exc)[:120]}")
    save_json(dict(model_kw=MODEL_KW, seeds=seeds, epochs=epochs), P["models"] / "config.json")
    return done


# ------------------------------------------------------------------------------------------------
# Loading members and predicting
# ------------------------------------------------------------------------------------------------
def _load_member(m, window, F, kw):
    if m["kind"] == "sklearn":
        import joblib
        return joblib.load(m["path"])
    from src import rul_models as RM
    return RM.load_model(m["arch"], window, F, m["path"], **kw)


def _predict_member(kind, model, X, cap=RUL_CAP):
    if kind == "sklearn":
        return np.clip(model.predict(flat_features(X)), 0, cap)
    if len(X) <= 32:
        out = np.asarray(model(np.asarray(X, dtype=np.float32), training=False).numpy())
    else:
        out = np.asarray(model.predict(X, batch_size=1024, verbose=0))
    return np.clip(out.ravel() * cap, 0, cap)


def discover_specs(root, ds):
    """Find every trained model on disk and describe it. Deep architectures that are not trained are listed as such."""
    P = paths(root, ds)
    specs = []
    for n in CLASSICAL:
        f = P["models"] / "classical" / f"{n}.joblib"
        if f.exists():
            specs.append(dict(id=n, name=PRETTY[n], family="classical", desc=DESC[n], status="ready",
                              members=[dict(kind="sklearn", path=str(f), weight=1.0)]))
    et, gb = (P["models"] / "classical" / f"{n}.joblib" for n in ("ExtraTrees", "HistGradientBoosting"))
    if et.exists() and gb.exists():
        specs.append(dict(id="classical_ensemble", name="Classical ensemble", family="ensemble", status="ready",
                          desc="Average of Extra Trees and Gradient Boosting.",
                          members=[dict(kind="sklearn", path=str(et), weight=0.5), dict(kind="sklearn", path=str(gb), weight=0.5)]))
    for a in DEEP_ARCHS:
        files = sorted(P["models"].glob(f"{a}_s*.weights.h5"))
        if files:
            specs.append(dict(id=a, name=PRETTY[a], family="deep", desc=DESC[a], status="ready",
                              members=[dict(kind="keras", arch=a, path=str(f), weight=1.0 / len(files)) for f in files]))
        else:
            specs.append(dict(id=a, name=PRETTY[a], family="deep", desc=DESC[a], status="not_trained", members=[]))
    return specs


def _metrics(y, p, y_raw=None):
    m = regression_metrics(y, p)
    out = {k: round(float(v), 4) for k, v in m.items()}
    if y_raw is not None:
        r = regression_metrics(y_raw, p)
        out["R2_raw"], out["RMSE_raw"], out["MAE_raw"] = round(float(r["R2"]), 4), round(float(r["RMSE"]), 4), round(float(r["MAE"]), 4)
    return out


def build_zoo(root, ds, log=print, train_sample=6000):
    """Evaluate every available model on identical data and write the files the web UI reads."""
    P = paths(root, ds)
    z = np.load(P["proc"] / "arrays.npz")
    prep = Preprocessor.from_dict(load_json(P["proc"] / "prep.json"))
    det = AnomalyDetector.from_dict(load_json(P["proc"] / "anomaly.json"))
    window, F, cap = z["X_train"].shape[1], z["X_train"].shape[2], RUL_CAP
    cfgp = P["models"] / "config.json"
    kw = load_json(cfgp).get("model_kw", MODEL_KW) if cfgp.exists() else MODEL_KW
    ensure_classical(root, ds, log)
    specs = discover_specs(root, ds)

    sens = pd.read_csv(P["final"] / "test_sensors.csv.gz")
    fleet0 = pd.read_csv(P["final"] / "fleet_summary.csv").set_index("unit_id")
    units = np.sort(sens["unit_id"].unique())
    rng = np.random.RandomState(0)
    idx = rng.choice(len(z["X_train"]), min(train_sample, len(z["X_train"])), replace=False)
    Xtr_s, ytr_s = z["X_train"][idx], z["y_train"][idx]

    # every cycle of every test engine (windows use only past data) + anomaly score
    Ws, rows = [], []
    for u in units:
        g = sens[sens.unit_id == u].sort_values("cycle")
        feat = prep.transform_engine(g)
        W = engine_windows(feat, window)
        Ws.append(W)
        last, true_last = int(g.cycle.max()), float(fleet0.loc[u, "rul_true"])
        true = true_last + (last - g.cycle.to_numpy())
        rows.append(pd.DataFrame({"unit_id": u, "cycle": g.cycle.to_numpy(), "rul_true": true, "rul_true_capped": np.minimum(true, cap),
                                  "anomaly_score": warmup_fix(det.score(prep.sensor_part(feat)))}))
    W_all, TL = np.concatenate(Ws), pd.concat(rows, ignore_index=True)
    X_last = np.stack([w[-1] for w in Ws])
    names = prep.features

    # per-member predictions (each member evaluated once, shared by every model that uses it)
    cache, mp = {}, {}
    def member_preds(m):
        key = m["path"]
        if key in mp:
            return mp[key]
        model = _load_member(m, window, F, kw)
        k = m["kind"]
        base = _predict_member(k, model, X_last)
        occ = np.zeros((len(X_last), F))
        for j in range(F):
            Xo = X_last.copy(); Xo[:, :, j] = 0.0
            occ[:, j] = _predict_member(k, model, Xo) - base
        t0 = time.perf_counter()
        for i in range(20):
            _predict_member(k, model, X_last[i:i + 1])
        lat = (time.perf_counter() - t0) / 20 * 1000
        params = int(model.count_params()) if k == "keras" else None
        mp[key] = dict(train=_predict_member(k, model, Xtr_s), val=_predict_member(k, model, z["X_val"]),
                       test=_predict_member(k, model, z["X_test"]), tl=_predict_member(k, model, W_all), occ=occ, lat=lat,
                       size=Path(m["path"]).stat().st_size / 1024, params=params)
        return mp[key]

    def combine(spec):
        out = dict(train=0, val=0, test=0, tl=0, occ=0, lat=0, size=0, params=0)
        tot = sum(m["weight"] for m in spec["members"])
        for m in spec["members"]:
            r, w = member_preds(m), m["weight"] / tot
            for k in ("train", "val", "test", "tl", "occ"):
                out[k] = out[k] + w * r[k]
            out["lat"] += r["lat"]; out["size"] += r["size"]
            out["params"] += (r["params"] or 0)
        return out

    results, deep_rank = {}, []
    ready = [s for s in specs if s["status"] == "ready"]
    for s in ready:
        results[s["id"]] = combine(s)
        log(f"[{ds}] evaluated {s['name']}")
    # ensembles of the best deep architectures (ranked on VALIDATION only)
    deep = sorted([s for s in ready if s["family"] == "deep"], key=lambda s: _metrics(z["y_val"], results[s["id"]]["val"])["RMSE"])
    if len(deep) >= 2:
        top = deep[:3]
        members = [dict(m, weight=m["weight"] / len(top)) for s in top for m in s["members"]]
        s3 = dict(id="deep_top3", name="Deep top-3 ensemble", family="ensemble", status="ready", members=members,
                  desc="Average of the 3 best deep architectures (chosen on validation engines): " + ", ".join(t["name"] for t in top))
        specs.append(s3); ready.append(s3); results[s3["id"]] = combine(s3)
        if any(s["id"] == "classical_ensemble" for s in ready):
            ce = next(s for s in ready if s["id"] == "classical_ensemble")
            sh = dict(id="hybrid", name="Hybrid: deep + classical", family="ensemble", status="ready",
                      members=[dict(m, weight=m["weight"] * 0.5) for m in s3["members"]] + [dict(m, weight=m["weight"] * 0.5) for m in ce["members"]],
                      desc="Half deep top-3 ensemble, half classical ensemble.")
            specs.append(sh); ready.append(sh); results[sh["id"]] = combine(sh)

    for u_i, s in enumerate(specs):
        if s["status"] != "ready":
            continue
        r = results[s["id"]]
        s["metrics"] = dict(train=_metrics(ytr_s, r["train"]), val=_metrics(z["y_val"], r["val"]),
                            test=_metrics(z["y_test"], r["test"], z["y_test_raw"]))
        s["size_kb"], s["latency_ms"] = round(r["size"], 1), round(r["lat"], 3)
        s["params"] = int(r["params"]) if r["params"] else None
        s["n_members"] = len(s["members"])
        TL[s["id"]] = np.round(r["tl"], 3)
    ok = [s for s in specs if s["status"] == "ready"]
    best = min(ok, key=lambda s: s["metrics"]["val"]["RMSE"])            # recommended = best on VALIDATION (test never used)
    for s in specs:
        s["recommended"] = s["id"] == best["id"]
        for m in s["members"]:
            m["path"] = str(Path(m["path"]).resolve().relative_to(Path(root).resolve())).replace("\\", "/")

    P["zoo"].mkdir(parents=True, exist_ok=True)
    imp = []
    for s in ok:
        occ = results[s["id"]]["occ"]
        imp.append(pd.DataFrame({"model": s["id"], "unit_id": np.repeat(units, F), "feature": np.tile(names, len(units)),
                                 "contribution": np.round(occ.ravel(), 4)}))
    pd.concat(imp, ignore_index=True).to_csv(P["zoo"] / "importance.csv.gz", index=False)
    TL.to_csv(P["zoo"] / "timelines.csv.gz", index=False)
    index = dict(dataset=ds, cap=cap, window=window, features=names, recommended=best["id"], deep_available=any(s["family"] == "deep" for s in ok),
                 tensorflow=has_tensorflow(), n_test_engines=int(len(units)), model_kw=kw, models=[
                     {k: v for k, v in s.items()} for s in specs])
    save_json(index, P["zoo"] / "index.json")
    log(f"[{ds}] model zoo written: {len(ok)} models ready, recommended = {best['name']}")
    return index


# ------------------------------------------------------------------------------------------------
# Live inference for the API (uploaded CSV / any machine history)
# ------------------------------------------------------------------------------------------------
class ZooPredictor:
    def __init__(self, root, ds, model_id=None):
        P = paths(root, ds)
        self.root, self.ds = Path(root), ds
        self.index = load_json(P["zoo"] / "index.json")
        self.prep = Preprocessor.from_dict(load_json(P["final"] / "prep.json"))
        self.det = AnomalyDetector.from_dict(load_json(P["final"] / "anomaly.json"))
        self.window, self.cap = self.index["window"], self.index["cap"]
        mid = model_id or self.index["recommended"]
        self.spec = next((m for m in self.index["models"] if m["id"] == mid), None)
        if self.spec is None or self.spec["status"] != "ready":
            raise ValueError(f"model '{mid}' is not available for {ds}")
        self._members = None

    def _load(self):
        if self._members is None:
            self._members = [(m["kind"], _load_member(dict(m, path=str(self.root / m["path"])), self.window, self.prep.n_features, self.index["model_kw"]),
                              m["weight"]) for m in self.spec["members"]]

    def predict_windows(self, X):
        self._load()
        tot = sum(w for _, _, w in self._members)
        return sum(w * _predict_member(k, m, X, self.cap) for k, m, w in self._members) / tot

    def timeline(self, df):
        g = df.sort_values("cycle")
        feat = self.prep.transform_engine(g)
        rul = self.predict_windows(engine_windows(feat, self.window))
        return pd.DataFrame({"cycle": g.cycle.to_numpy(), "rul_pred": rul,
                             "anomaly_score": warmup_fix(self.det.score(self.prep.sensor_part(feat)))}), feat

    def occlusion(self, window_last):
        base = self.predict_windows(window_last[None])[0]
        out = []
        for j in range(window_last.shape[1]):
            x = window_last.copy(); x[:, j] = 0.0
            out.append(float(self.predict_windows(x[None])[0] - base))
        return out

In [ ]:
import sys, os, json, time, shutil, warnings, random
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import joblib
warnings.filterwarnings("ignore")

ROOT = Path.cwd(); sys.path.insert(0, str(ROOT))
DATA_DIR = ROOT / "data" / "raw"; PROC = ROOT / "data" / "processed"
MODELS = ROOT / "artifacts" / "models"; FINAL = ROOT / "artifacts" / "final"; OUT = ROOT / "artifacts" / "phase3"
for d in (MODELS, FINAL, OUT): d.mkdir(parents=True, exist_ok=True)
try: display
except NameError: display = print

from src.rul_core import *
from src import rul_models as RM
from src.inference import Predictor, export_dashboard_data
from src import viz

# ======================= CONFIGURATION =======================
DATASETS = ["FD001", "FD004"]     # must have been prepared in Phase 2
QUICK_RUN = False                 # True = 5-minute smoke test (1 seed, few epochs). False = full experiment.
ARCHS = ["mlp", "lstm", "gru", "bilstm", "cnn_lstm", "cnn_lstm_attention"]
N_SEEDS = 1 if QUICK_RUN else 3   # seeds per architecture (ensemble members)
EPOCHS = 6 if QUICK_RUN else 60   # upper limit; early stopping normally stops much earlier
PATIENCE = 3 if QUICK_RUN else 10
BATCH = 128
MODEL_KW = dict(units=64, filters=64, dropout=0.2, l2=1e-5, noise=0.03, lr=1e-3)
TARGET_R2 = 0.75
# =============================================================
np.random.seed(SEED); random.seed(SEED)
print("TensorFlow:", RM.TF_VERSION, "| GPUs:", (RM.tf.config.list_physical_devices("GPU") if RM.HAS_TF else "n/a"))
if not RM.HAS_TF:
    print("\n*** TensorFlow is NOT installed: deep models will be skipped and the best classical model is exported. Run `pip install tensorflow` and re-run for the full deep-learning results. ***")

## 1. Load the Phase-2 tensors

In [ ]:
R = {}
for c in DATASETS:
    z = np.load(PROC / c / "arrays.npz")
    R[c] = dict(z={k: z[k] for k in z.files}, prep=Preprocessor.from_dict(load_json(PROC / c / "prep.json")),
                det=AnomalyDetector.from_dict(load_json(PROC / c / "anomaly.json")), members={}, hist={})
    R[c]["w"], R[c]["F"] = R[c]["z"]["X_train"].shape[1], R[c]["z"]["X_train"].shape[2]
    print(f"{c}: train {R[c]['z']['X_train'].shape} | val {R[c]['z']['X_val'].shape} | test {R[c]['z']['X_test'].shape} | features: {R[c]['prep'].features}")

## 2. Evaluation metrics — what they mean
| Metric | Meaning | Better |
|---|---|---|
| **MAE** | average absolute error in cycles | lower |
| **RMSE** | like MAE but punishes large errors | lower |
| **R²** | share of RUL variance explained (1 = perfect, 0 = predicting the mean) | higher |
| **NASA score** | asymmetric: *late* predictions (dangerous) cost more than early ones | lower |
| **Within ±10** | % of engines predicted within 10 cycles | higher |

*Validation* = all windows of 20 % held-out training engines (RUL capped at 125, the same target the model learns).
*Test* = the **official** protocol: the last window of every test engine vs NASA's RUL file. We report test R² against the **capped** RUL
(like training) and, to be fully transparent, also against the **raw uncapped** RUL file.

In [ ]:
def metric_row(name, kind, c, pv, pt):
    z = R[c]["z"]; v = regression_metrics(z["y_val"], pv); t = regression_metrics(z["y_test"], pt); r = regression_metrics(z["y_test_raw"], pt)
    return {"Model": name, "Type": kind, "Val_RMSE": v["RMSE"], "Val_MAE": v["MAE"], "Val_R2": v["R2"],
            "Test_RMSE": t["RMSE"], "Test_MAE": t["MAE"], "Test_R2": t["R2"], "Test_R2_rawRUL": r["R2"], "Test_RMSE_rawRUL": r["RMSE"],
            "Test_MAE_rawRUL": r["MAE"], "NASA_score": r["NASA_score"], "Within_10_%": r["Within_10_cycles_%"]}

def member_predict(m, X, cap=RUL_CAP):
    return np.clip(m["model"].predict(flat_features(X)), 0, cap) if m["kind"] == "sklearn" else RM.predict_rul(m["model"], X, cap)

## 3. Classical (non-deep) baselines
Input = 7 statistics per sensor over the window (last value, mean, std, min, max, slope, mean of last 5). These show how far *classical ML* gets, so the deep models have to earn their place.

In [ ]:
for c in DATASETS:
    r, z = R[c], R[c]["z"]
    Ftr, Fva, Fte = (flat_features(z[k]) for k in ("X_train", "X_val", "X_test"))
    t0 = time.time(); models = RM.fit_classical(Ftr, z["y_train"], seed=0, fast=QUICK_RUN)
    for name, m in models.items():
        r["members"][f"sk:{name}"] = dict(kind="sklearn", name=name, model=m, val=np.clip(m.predict(Fva), 0, RUL_CAP), test=np.clip(m.predict(Fte), 0, RUL_CAP))
    print(f"{c}: classical models trained in {time.time() - t0:.0f}s")
    display(pd.DataFrame([metric_row(m["name"], "classical", c, m["val"], m["test"]) for m in r["members"].values()]).round(3).iloc[:, :8])

## 4. Deep models (Keras)
Input window `(time-steps × features)` → network → one number = RUL/125. Loss = Huber (robust to outliers). Adam optimiser, gradient clipping.

* **MLP** – no memory of time order (control experiment).
* **LSTM / GRU / BiLSTM** – recurrent layers read the window step by step.
* **CNN-LSTM** – 1D convolutions extract local sensor patterns, the LSTM models how they evolve.
* **CNN-BiLSTM-Attention** – adds *temporal attention*: the model learns **which time steps matter** for the current RUL.

Each architecture is trained with several seeds; their average (an *ensemble*) is more accurate and more stable than any single run.

In [ ]:
DEEP_OK = {c: False for c in DATASETS}
if RM.HAS_TF:
    for c in DATASETS:
        r, z = R[c], R[c]["z"]
        for arch in ARCHS:
            t0 = time.time()
            try:
                for s in range(N_SEEDS):
                    RM.set_seed(1000 * (ARCHS.index(arch) + 1) + s)
                    model = RM.build_model(arch, r["w"], r["F"], **MODEL_KW)
                    h = RM.train_model(model, z["X_train"], z["y_train"], z["X_val"], z["y_val"], RUL_CAP, EPOCHS, BATCH, PATIENCE, verbose=0)
                    wpath = RM.save_weights(model, MODELS / c / f"{arch}_s{s}")
                    r["members"][f"keras:{arch}:s{s}"] = dict(kind="keras", name=arch, arch=arch, seed=s, model=model, file=wpath,
                                                              val=RM.predict_rul(model, z["X_val"]), test=RM.predict_rul(model, z["X_test"]))
                    if s == 0: r["hist"][arch] = {k: [float(v) for v in vals] for k, vals in h.history.items()}
                DEEP_OK[c] = True
                v = np.mean([r["members"][f"keras:{arch}:s{s}"]["val"] for s in range(N_SEEDS)], axis=0)
                print(f"{c} | {arch:20s} params={model.count_params():>7,} epochs={len(h.history['loss']):>3} | ensemble val RMSE={regression_metrics(z['y_val'], v)['RMSE']:.2f} | {time.time() - t0:.0f}s")
            except Exception as exc:                      # one failing model must never stop the whole experiment
                print(f"{c} | {arch}: FAILED ({type(exc).__name__}: {str(exc)[:150]}) - skipped")
else:
    print("TensorFlow not available -> skipping deep models.")

In [ ]:
hist_c = DATASETS[0]
if R[hist_c]["hist"]:
    fig = viz.fig_training_curves(R[hist_c]["hist"], FINAL / hist_c / "figures" / "learning_curves.png"); plt.show()
    gaps = {a: h["val_loss"][-1] / max(h["loss"][-1], 1e-9) for a, h in R[hist_c]["hist"].items()}
    print("final val_loss / train_loss ratio per model (≈1-3 = healthy; >>5 would signal over-fitting):", {k: round(v, 2) for k, v in gaps.items()})

## 5. Model selection — on VALIDATION only
Candidates: every single classical model, every deep architecture (seed-ensemble), a **Deep Top-3 ensemble**, a **Classical ensemble** and a **Hybrid** (deep + classical).
We pick the lowest *validation* RMSE. The test set is not touched for this decision, so the test score is an unbiased estimate.

In [ ]:
def blend(c, weights):
    tot = sum(weights.values()); m = R[c]["members"]
    return (sum(w * m[k]["val"] for k, w in weights.items()) / tot, sum(w * m[k]["test"] for k, w in weights.items()) / tot)

CAND = {}
for c in DATASETS:
    m = R[c]["members"]; cand = {}
    for k, v in m.items():
        if v["kind"] == "sklearn": cand[v["name"]] = ("classical", {k: 1.0})
    for arch in ARCHS:
        ks = [k for k, v in m.items() if v["kind"] == "keras" and v["arch"] == arch]
        if ks: cand[f"{arch} (x{len(ks)} seeds)"] = ("deep", {k: 1.0 for k in ks})
    cls = {k: 1.0 for k in ("sk:ExtraTrees", "sk:HistGradientBoosting")}
    cand["Classical ensemble (ExtraTrees+GBM)"] = ("ensemble", cls)
    deep_rank = sorted([(regression_metrics(R[c]["z"]["y_val"], blend(c, w)[0])["RMSE"], n) for n, (t, w) in cand.items() if t == "deep"])
    if len(deep_rank) >= 2:
        top = [n for _, n in deep_rank[:3]]
        wdeep = {k: 1.0 / len(cand[n][1]) / len(top) for n in top for k in cand[n][1]}
        cand["Deep Top-3 ensemble"] = ("ensemble", wdeep)
        wh = {k: 0.5 * w for k, w in wdeep.items()}; wh.update({k: 0.25 for k in cls})
        cand["Hybrid: Deep Top-3 + Classical"] = ("ensemble", wh)
    CAND[c] = cand

TABLE, CHOSEN = {}, {}
for c in DATASETS:
    rows = [metric_row(n, t, c, *blend(c, w)) for n, (t, w) in CAND[c].items()]
    TABLE[c] = pd.DataFrame(rows).sort_values("Val_RMSE").reset_index(drop=True)
    CHOSEN[c] = TABLE[c].iloc[0]["Model"]
    print(f"\n===== {c}: candidates ranked by VALIDATION RMSE =====")
    display(TABLE[c].round(3))
    print(f"-> selected on validation: {CHOSEN[c]}")

## 6. Official test results

In [ ]:
for c in DATASETS:
    t = TABLE[c]; ch = t[t.Model == CHOSEN[c]].iloc[0]
    print(f"{c}  SELECTED MODEL: {CHOSEN[c]}")
    print(f"   validation : R² = {ch.Val_R2:.3f}  RMSE = {ch.Val_RMSE:.2f}  MAE = {ch.Val_MAE:.2f}")
    print(f"   TEST       : R² = {ch.Test_R2:.3f}  RMSE = {ch.Test_RMSE:.2f}  MAE = {ch.Test_MAE:.2f}      (target: capped RUL, same as training)")
    print(f"   TEST (raw) : R² = {ch.Test_R2_rawRUL:.3f}  RMSE = {ch.Test_RMSE_rawRUL:.2f}  MAE = {ch.Test_MAE_rawRUL:.2f}   NASA score = {ch.NASA_score:.0f}   within ±10 cycles = {ch['Within_10_%']:.0f}%")
    (FINAL / c / "figures").mkdir(parents=True, exist_ok=True)
    viz.fig_model_comparison(t, "Test_R2", f"{c}: official test R² (higher is better)", False, CHOSEN[c], FINAL / c / "figures" / "model_comparison_r2.png"); plt.show()
    viz.fig_model_comparison(t, "Test_RMSE", f"{c}: official test RMSE in cycles (lower is better)", True, CHOSEN[c], FINAL / c / "figures" / "model_comparison_rmse.png"); plt.show()
    w = CAND[c][CHOSEN[c]][1]; pt = blend(c, w)[1]
    viz.fig_pred_vs_true(R[c]["z"]["y_test"], pt, f"{c}: predicted vs actual RUL - {CHOSEN[c]}", FINAL / c / "figures" / "pred_vs_true.png"); plt.show()

## 7. Over-fitting check
If a model memorised the training engines, its error on training data would be far smaller than on unseen data.
We compare RMSE on (a) a sample of training windows, (b) the held-out validation engines, (c) the official test engines.

In [ ]:
OVERFIT = {}
for c in DATASETS:
    z, w = R[c]["z"], CAND[c][CHOSEN[c]][1]; rng = np.random.RandomState(0)
    idx = rng.choice(len(z["X_train"]), min(6000, len(z["X_train"])), replace=False)
    tot = sum(w.values()); ptr = sum(wt * member_predict(R[c]["members"][k], z["X_train"][idx]) for k, wt in w.items()) / tot
    pv, pt = blend(c, w)
    OVERFIT[c] = dict(train_RMSE=regression_metrics(z["y_train"][idx], ptr)["RMSE"], val_RMSE=regression_metrics(z["y_val"], pv)["RMSE"], test_RMSE=regression_metrics(z["y_test"], pt)["RMSE"],
                      train_R2=regression_metrics(z["y_train"][idx], ptr)["R2"], val_R2=regression_metrics(z["y_val"], pv)["R2"], test_R2=regression_metrics(z["y_test"], pt)["R2"])
    o = OVERFIT[c]; print(f"{c}: RMSE train {o['train_RMSE']:.1f} | val {o['val_RMSE']:.1f} | test {o['test_RMSE']:.1f}      R² train {o['train_R2']:.3f} | val {o['val_R2']:.3f} | test {o['test_R2']:.3f}")
    gap = o["val_R2"] - o["test_R2"]
    print(f"   held-out validation vs held-out test: ΔR² = {gap:+.3f}  ->",
          "CONSISTENT on unseen engines: no harmful over-fitting to the validation set or the training engines" if abs(gap) < 0.08 and o["val_R2"] >= TARGET_R2
          else "WARNING: validation and test disagree - use more seeds / stronger regularisation")
    print("   (training-set R² is always optimistic - especially for tree ensembles; the honest evidence is the UNSEEN-engine scores above.)")

## 8. Anomaly detection (unsupervised) — precision / recall / F1
The Mahalanobis detector from Phase 2 was fitted on **healthy early-life data only**. To score it we define *"degraded" = true RUL ≤ 50 cycles* and ask:
does the anomaly alarm (score > 0.5) fire on exactly those engines?

In [ ]:
ANOM = {}
for c in DATASETS:
    r, z = R[c], R[c]["z"]; ns = len(r["prep"].sensors); out = {}
    for split, X, y in (("validation", z["X_val"], z["y_val_raw"]), ("test", z["X_test"], z["y_test_raw"])):
        s = r["det"].score(X[:, -1, :ns]); lab = (y <= 50).astype(int)
        out[split] = classification_metrics(lab, (s > 0.5).astype(int), s)
    ANOM[c] = out; print(c, {k: {m: round(v, 3) for m, v in d.items()} for k, d in out.items()})

## 9. Export the final model + dashboard data
Saves the chosen ensemble (weights), preprocessing, anomaly detector and per-engine timelines, so the **backend and dashboard need no retraining**.

In [ ]:
INFO = {}
for c in DATASETS:
    r, out = R[c], FINAL / c
    (out / "members").mkdir(parents=True, exist_ok=True); (out / "figures").mkdir(exist_ok=True)
    members = []; w = CAND[c][CHOSEN[c]][1]; tot = sum(w.values())
    for k, wt in w.items():
        m = r["members"][k]
        if m["kind"] == "sklearn":
            fname = f"members/{m['name']}.joblib"; joblib.dump(m["model"], out / fname, compress=3); members.append(dict(kind="sklearn", file=fname, weight=wt / tot))
        else:
            fname = f"members/{m['arch']}_s{m['seed']}.weights.h5"; shutil.copy(m["file"], out / fname); members.append(dict(kind="keras", arch=m["arch"], file=fname, kw=MODEL_KW, weight=wt / tot))
    ch = TABLE[c][TABLE[c].Model == CHOSEN[c]].iloc[0].to_dict()
    meta = dict(dataset=c, window=r["w"], cap=RUL_CAP, features=r["prep"].features, description=DATASET_CFG[c]["description"],
                final_model=dict(name=CHOSEN[c], members=members), has_tensorflow=RM.HAS_TF, tf_version=RM.TF_VERSION, quick_run=QUICK_RUN)
    save_json(meta, out / "meta.json"); save_json(r["prep"].to_dict(), out / "prep.json"); save_json(r["det"].to_dict(), out / "anomaly.json")
    pred = Predictor(out)
    chk = pred.predict_windows(r["z"]["X_test"]); ref = blend(c, w)[1]
    assert np.allclose(chk, ref, atol=0.5), f"reloaded model differs from in-memory model (max diff {np.abs(chk - ref).max():.3f})"
    print(f"{c}: exported '{CHOSEN[c]}' ({len(members)} members); reload check OK (max diff {np.abs(chk - ref).max():.4f} cycles)")
    test_df = load_subset(find_cmapss_files(DATA_DIR), c)[1]
    D = export_dashboard_data(pred, test_df, load_subset(find_cmapss_files(DATA_DIR), c)[2], out)
    INFO[c] = dict(pred=pred, D=D, meta=meta, chosen=ch)
    print(f"   dashboard data: {len(D['fleet'])} engines, {len(D['timelines']):,} timeline rows")

## 9b. Model zoo for the web app
Every trained model (Keras networks, their ensembles and the classical models) is scored on the **same** engines and exported, so the web app can compare them side by side.

In [ ]:
from src import zoo
for c in DATASETS:
    zoo.build_zoo(ROOT, c, log=print)

## 10. Explainable AI — *why* does the model predict a short RUL?
Two complementary methods:
* **Occlusion** (model-agnostic): reset one sensor to its healthy value and see how many cycles the predicted RUL *rises*. Large rise ⇒ that sensor is pushing the RUL down.
* **Integrated Gradients** (Keras, gradient-based): attributes the prediction to every (time-step, sensor) input value.

In [ ]:
XAI = {}
for c in DATASETS:
    pred, r = INFO[c]["pred"], R[c]; z = r["z"]; names = r["prep"].features
    occ = pred.occlusion_importance(z["X_test"])
    fi_occ = pd.Series(np.abs(occ).mean(0), index=names).sort_values(ascending=False)
    XAI[c] = dict(occlusion=fi_occ.to_dict())
    deep = [k for k, v in r["members"].items() if v["kind"] == "keras"]
    if deep:
        best = min(deep, key=lambda k: regression_metrics(z["y_val"], r["members"][k]["val"])["RMSE"])
        ig = RM.integrated_gradients(r["members"][best]["model"], z["X_test"], steps=24)
        fi_ig = pd.Series(np.abs(ig).sum(1).mean(0), index=names).sort_values(ascending=False)
        XAI[c]["integrated_gradients"] = fi_ig.to_dict()
        fig, axs = plt.subplots(1, 3, figsize=(16, 4.2), gridspec_kw={"width_ratios": [1.2, 1, 1]})
        im = axs[0].imshow(np.abs(ig).mean(0).T, aspect="auto", cmap="magma"); axs[0].set_yticks(range(len(names))); axs[0].set_yticklabels(names, fontsize=7); axs[0].grid(False)
        axs[0].set_xlabel("time step in window (right = most recent)"); axs[0].set_title("Integrated Gradients: where does the model look?"); plt.colorbar(im, ax=axs[0], fraction=0.04)
        top = fi_ig.head(8)[::-1]; axs[1].barh(top.index, top.values, color=viz.VIOLET); axs[1].set_title("Top sensors - Integrated Gradients")
        top = fi_occ.head(8)[::-1]; axs[2].barh(top.index, top.values, color=viz.BLUE); axs[2].set_title("Top sensors - occlusion (cycles)")
        fig.tight_layout(); fig.savefig(FINAL / c / "figures" / "explainability.png", bbox_inches="tight"); plt.show()
        from scipy.stats import spearmanr
        print(f"{c}: rank agreement between the two explanation methods (Spearman) = {spearmanr(fi_ig[names].values, fi_occ[names].values)[0]:.2f}")
    else:
        top = fi_occ.head(8)[::-1]; fig, ax = plt.subplots(figsize=(6, 3.6)); ax.barh(top.index, top.values, color=viz.BLUE); ax.set_title(f"{c}: top sensors (occlusion)")
        fig.savefig(FINAL / c / "figures" / "explainability.png", bbox_inches="tight"); plt.show()
    print(f"{c}: most influential inputs:", list(fi_occ.head(5).index))

In [ ]:
c = DATASETS[0]; D = INFO[c]["D"]; fleet = D["fleet"]
u = int(fleet.sort_values("rul_pred").iloc[len(fleet) // 4]["unit_id"])           # an engine in the 'warning' zone
tl = D["timelines"][D["timelines"].unit_id == u]; imp = D["importance"][D["importance"].unit_id == u]
viz.fig_rul_timeline(tl.cycle, tl.rul_pred, tl.rul_true_capped, RUL_CAP, title=f"{c} - test engine {u}: predicted vs true RUL over its life"); plt.show()
viz.fig_anomaly_timeline(tl.cycle, tl.anomaly_score, title=f"{c} - engine {u}: anomaly score"); plt.show()
viz.fig_importance(imp.label + " " + imp.trend, imp.contribution, title=f"Engine {u}: why is the RUL low?"); plt.show()

## 11. Model compression for edge / IoT deployment
* **Knowledge distillation:** a tiny GRU student (16 units) learns from the big model's predictions (70 %) + ground truth (30 %).
* **Quantisation:** float16 weights (always possible) and TensorFlow-Lite dynamic-range int8 (when the converter supports the LSTM ops on your machine).

We report size, per-window latency and accuracy, and only claim a benefit if the error stays acceptable.

In [ ]:
COMP = {}
if RM.HAS_TF:
    for c in DATASETS:
        r, z = R[c], R[c]["z"]; deep = [k for k, v in r["members"].items() if v["kind"] == "keras"]
        if not deep: continue
        try:
            best = min(deep, key=lambda k: regression_metrics(z["y_val"], r["members"][k]["val"])["RMSE"]); teacher = r["members"][best]["model"]; arch = r["members"][best]["arch"]
            soft = teacher.predict(z["X_train"], batch_size=1024, verbose=0).ravel()
            student = RM.distill_student(soft, z["X_train"], z["y_train"], z["X_val"], z["y_val"], RUL_CAP, epochs=10 if QUICK_RUN else 40)
            f16, f16_bytes = RM.float16_quantize(teacher, arch, r["w"], r["F"], **MODEL_KW)
            rows = []
            def add(name, model, size, lat_fn=None, preds=None):
                p = preds if preds is not None else RM.predict_rul(model, z["X_test"]); lat = RM.latency_ms(lat_fn or (lambda x: model(x, training=False).numpy()), z["X_test"])
                rows.append(dict(Model=name, Size_KB=size / 1024, Latency_ms=lat, Test_RMSE=regression_metrics(z["y_test"], p)["RMSE"], Test_R2=regression_metrics(z["y_test"], p)["R2"]))
            add(f"Teacher ({arch}, float32)", teacher, teacher.count_params() * 4)
            add("Teacher float16 weights", f16, f16_bytes)
            add("Distilled student (tiny GRU)", student, student.count_params() * 4)
            size, info = RM.tflite_dynamic_quantize(teacher, MODELS / c / "teacher_dynamic_int8.tflite")
            if size:
                try:
                    p = RM.tflite_predict(info, z["X_test"]); add("Teacher TFLite int8 (dynamic)", None, size, lat_fn=lambda x: RM.tflite_predict(info, x), preds=p)
                except Exception as exc: print("TFLite model created but could not be executed here:", str(exc)[:100]); rows.append(dict(Model="Teacher TFLite int8 (size only)", Size_KB=size / 1024, Latency_ms=np.nan, Test_RMSE=np.nan, Test_R2=np.nan))
            else: print(f"{c}: TFLite conversion not available here ({info}); float16 quantisation is reported instead.")
            COMP[c] = pd.DataFrame(rows); display(COMP[c].round(3))
            fig, axs = plt.subplots(1, 3, figsize=(13, 3.2))
            for ax, col in zip(axs, ["Size_KB", "Latency_ms", "Test_RMSE"]): ax.barh(COMP[c]["Model"], COMP[c][col], color=viz.BLUE); ax.set_title(col)
            axs[1].set_yticklabels([]); axs[2].set_yticklabels([]); fig.tight_layout(); fig.savefig(FINAL / c / "figures" / "compression.png", bbox_inches="tight"); plt.show()
        except Exception as exc:
            print(f"{c}: compression step skipped ({type(exc).__name__}: {str(exc)[:150]})")
else:
    print("TensorFlow not available: compression needs the Keras models. Skipped.")

## 12. Save results & final summary

In [ ]:
viz.fig_pipeline(FINAL / "pipeline.png"); plt.show()
index = {"datasets": [], "default": DATASETS[0]}
for c in DATASETS:
    t = TABLE[c]; ch = INFO[c]["chosen"]
    res = dict(dataset=c, description=DATASET_CFG[c]["description"], selected_model=CHOSEN[c], has_tensorflow=RM.HAS_TF, tf_version=RM.TF_VERSION, quick_run=QUICK_RUN,
               rul_cap=RUL_CAP, window=R[c]["w"], n_features=R[c]["F"], headline={k: float(v) for k, v in ch.items() if k not in ("Model", "Type")},
               table=t.round(4).to_dict("records"), overfit_check=OVERFIT[c], anomaly=ANOM[c], explainability=XAI[c],
               compression=(COMP[c].round(4).to_dict("records") if c in COMP else []),
               target_r2=TARGET_R2, target_met=bool(ch["Test_R2"] >= TARGET_R2 and ch["Val_R2"] >= TARGET_R2))
    save_json(res, FINAL / c / "results.json"); index["datasets"].append(c)
save_json(index, FINAL / "index.json")

print("=" * 78); print("FINAL SUMMARY"); print("=" * 78)
for c in DATASETS:
    ch = INFO[c]["chosen"]
    print(f"\n{c} ({DATASET_CFG[c]['description']})  ->  {CHOSEN[c]}")
    print(f"   Validation R²={ch['Val_R2']:.3f} | Test R² (capped RUL)={ch['Test_R2']:.3f} | Test R² (raw RUL)={ch['Test_R2_rawRUL']:.3f}")
    print(f"   Test MAE={ch['Test_MAE']:.2f} cycles | RMSE={ch['Test_RMSE']:.2f} cycles | NASA score={ch['NASA_score']:.0f} | within ±10 cycles: {ch['Within_10_%']:.0f}%")
    print(f"   Anomaly detection (test): precision={ANOM[c]['test']['Precision']:.2f} recall={ANOM[c]['test']['Recall']:.2f} F1={ANOM[c]['test']['F1']:.2f}")
    print(f"   Target R² >= {TARGET_R2}: {'PASS ✔' if ch['Test_R2'] >= TARGET_R2 and ch['Val_R2'] >= TARGET_R2 else 'NOT MET ✘ (see tips in README)'}")
print("\nPHASE 3 COMPLETE ✔  — launch the web app:   python run.py")